# Lesson 1.1: Raw HTTP, then the SDK (messages and roles)
*Build an AI Agent from Scratch with LangGraph · Chapter 1*

**Goal:** see that every model call is a list of messages in and one message out, send it once as raw HTTP and once
with the SDK, and learn where a conversation really lives.

**What you will be able to do**
- build a chat request body by hand: model, messages with roles, temperature
- send it with `httpx`, then read `choices[0].message.content` and `usage`
- send the same call with the `openai` SDK and get the same JSON back as a typed object
- keep a conversation in your own Python list, because the API remembers nothing

## How to use this notebook
- Run the cells **from top to bottom** (Shift + Enter). A cell can use names made in cells above it.
- Cells that start with `# calls the API` send a real request to Groq. There are five of them, each tiny.
  Without a key they print `skipped` and the notebook goes on with the numbers from the video.
- Your answers from the model may be worded a little differently from the video: that is normal (lesson 1.2 shows why).
- Each exercise has a starter cell with `# YOUR CODE HERE` and a **check cell** that prints `✓ Exercise N passed`.
  The check cells never call the API. Solutions are at the very end.
- **Colab:** upload the `nightwatch/` folder too, and set the key without typing it into code:
  `from getpass import getpass; os.environ["GROQ_API_KEY"] = getpass("Groq key: ")`.

In [1]:
# Setup: run this cell first.
import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))          # finds a .env in this folder or any folder above it
OFFLINE = os.environ.get("NOTEBOOK_OFFLINE") == "1"   # set NOTEBOOK_OFFLINE=1 to skip every API cell
HAVE_KEY = bool(os.environ.get("GROQ_API_KEY")) and not OFFLINE
MODEL = "openai/gpt-oss-20b"
if HAVE_KEY:
    print("GROQ_API_KEY found: cells marked 'calls the API' will really call Groq.")
else:
    print("No GROQ_API_KEY: cells marked 'calls the API' print 'skipped' and the video's numbers are used.")
    print("Get the free key in lesson 0.2, put the line GROQ_API_KEY=... in a .env file, restart the kernel.")

GROQ_API_KEY found: cells marked 'calls the API' will really call Groq.


In [2]:
# Make the course code importable: find the repo's nightwatch/ folder above this notebook.
import sys
from pathlib import Path

here = Path.cwd()
NIGHTWATCH = next((p / "nightwatch" for p in [here, *here.parents]
                   if (p / "nightwatch" / "agent_v0.py").exists()), None)
if NIGHTWATCH:
    sys.path.insert(0, str(NIGHTWATCH))
    print("nightwatch/ found at", Path(os.path.relpath(NIGHTWATCH, here)).as_posix())
else:
    print("nightwatch/ not found: open this notebook from lessons/<lesson>/notebook/ inside the course repo.")

nightwatch/ found at ../../../nightwatch


## 1. Recall (lesson 0.2): where the key lives
The key lives in the `.env` file, your code reads it from the environment, and it never appears in the code.
The cell below only says *whether* a key was loaded: it never prints the key.

In [3]:
key = os.environ.get("GROQ_API_KEY", "")
print("key loaded:", bool(key))

key loaded: True


## 2. Key words

| Word | Meaning |
|---|---|
| message | one turn of the conversation: a small dict with `role` and `content` |
| role | who is speaking: `system`, `user` or `assistant` |
| JSON | text that holds dicts and lists, so any program can read it |
| request body | the JSON we send: the model name and the list of messages (plus options) |
| response | the JSON that comes back |
| SDK | a ready-made library that builds and sends these requests for you |

## 3. The idea and a hand trace
Every call has one shape: **a list of messages goes in, one message comes out**, always with the role `assistant`.

| Role | Who writes it | Job |
|---|---|---|
| `system` | you, the developer | the rules for the whole conversation |
| `user` | the person or the program asking | the question or task (here: an alert from the shop) |
| `assistant` | the model | its reply; every reply comes back with this role |

**The API is stateless.** Think of a pen pal who forgets everything right after replying: every letter must carry the
whole conversation so far. The conversation lives in *your* program, as a Python list.

Hand trace of the hook (real run from the video, temperature 0):

| Call | Messages sent | Roles | Reply |
|---|---|---|---|
| 1 | 1 | user ("payments is failing, remember that") | "Got it, **payments** ..." |
| 2 | 1 | user ("which service is failing?") only | `unknown` |
| 3 | 3 | user, assistant (reply 1), user (question) | `payments` |

The model did not remember in call 3: **we reminded it** by sending the history.

## 4. Write it yourself
### 4.1 The request body, by hand
Plain JSON: one dict with three keys. Predict: how many messages, and which roles?

In [4]:
import json

SYSTEM = "You are Nightwatch, the on-call assistant for the shop Brightcart. Answer in one sentence."
ALERT = "Alert: checkout error rate is 23%. What does that mean?"

body = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": ALERT},
    ],
    "temperature": 0,
}
print(json.dumps(body, indent=2))

{
  "model": "openai/gpt-oss-20b",
  "messages": [
    {
      "role": "system",
      "content": "You are Nightwatch, the on-call assistant for the shop Brightcart. Answer in one sentence."
    },
    {
      "role": "user",
      "content": "Alert: checkout error rate is 23%. What does that mean?"
    }
  ],
  "temperature": 0
}


### 4.2 One POST with `httpx`
The key goes into an `Authorization` header, the body goes as JSON. `r.json()` turns the reply into a plain dict.

If you have no key, the cell uses `VIDEO_RESPONSE`: the same fields as the reply in the video, so the next cells work.

In [5]:
# The reply from the video (real run), trimmed to the fields this lesson reads.
VIDEO_RESPONSE = {
    "id": "(video run)", "object": "chat.completion", "model": MODEL,
    "choices": [{"index": 0, "finish_reason": "stop",
                 "message": {"role": "assistant",
                             "content": "It means that nearly one in four checkout attempts are failing, "
                                        "indicating a significant issue that needs immediate investigation."}}],
    "usage": {"prompt_tokens": 108, "completion_tokens": 102, "total_tokens": 210},
}

In [6]:
# calls the API (1 request)
import httpx

URL = "https://api.groq.com/openai/v1/chat/completions"

if HAVE_KEY:
    r = httpx.post(URL, json=body, timeout=60,
                   headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"})
    print("status:", r.status_code)
    data = r.json()
else:
    print("skipped (no key): using VIDEO_RESPONSE")
    data = VIDEO_RESPONSE
print("top-level keys:", list(data))

status: 200
top-level keys: ['id', 'object', 'created', 'model', 'choices', 'usage', 'usage_breakdown', 'system_fingerprint', 'x_groq', 'service_tier']


### 4.3 Read the response
Today we need two keys: `choices` (a list of possible replies; item 0 holds the message) and `usage` (tokens).

In [7]:
msg = data["choices"][0]["message"]
print("role:", msg["role"])
print("content:", msg["content"])
print("finish_reason:", data["choices"][0]["finish_reason"])
u = data["usage"]
print("usage:", {k: u[k] for k in ("prompt_tokens", "completion_tokens", "total_tokens")})
print("prompt + completion == total:", u["prompt_tokens"] + u["completion_tokens"] == u["total_tokens"])

role: assistant
content: It means that nearly one in four checkout attempts are failing, indicating a significant issue that needs immediate investigation.
finish_reason: stop
usage: {'prompt_tokens': 108, 'completion_tokens': 102, 'total_tokens': 210}
prompt + completion == total: True


In the video: status 200, ten top-level keys, `finish_reason` = `stop`, and usage **108 + 102 = 210** tokens.
102 completion tokens for one short sentence looks like a lot: lesson 1.2 shows where they go.

### 4.4 The history demo from the hook
A tiny helper `ask` sends a list and returns the reply's text. Then the three calls from the hand trace.

In [8]:
# calls the API (3 requests)
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("GROQ_API_KEY", "none"),
                base_url="https://api.groq.com/openai/v1", max_retries=6)

def ask(messages):
    resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
    return resp.choices[0].message.content

first = [{"role": "user", "content": "The failing service is called payments. Remember that."}]
second = [{"role": "user", "content": "Which service did I say is failing? Answer with one word, or 'unknown'."}]

if HAVE_KEY:
    a1 = ask(first)
    print("call 1 reply:", a1[:90])
    print("call 2, new list only:", ask(second))
else:
    print("skipped (no key): video replies used")
    a1 = "Got it, **payments** is the service that's failing."
history = first + [{"role": "assistant", "content": a1}] + second
print("call 3 sends", len(history), "messages, roles:", [m["role"] for m in history])
if HAVE_KEY:
    print("call 3, whole history:", ask(history))

call 1 reply: Got it—**payments** is the service that’s failing. Let me know what you’d like to do next 


call 2, new list only: unknown
call 3 sends 3 messages, roles: ['user', 'assistant', 'user']


call 3, whole history: payments


Video: call 2 alone said `unknown`; call 3 with three messages of history said `payments`.

## 5. The same call with the SDK + check
The `openai` package, pointed at Groq's address, sends the same JSON. What comes back is a typed `ChatCompletion`
object: we read it with dots instead of brackets.

In [9]:
import openai
print("openai", openai.__version__, "| httpx", httpx.__version__)

openai 3.51.0 | httpx 0.28.1


In [10]:
# calls the API (1 request)
if HAVE_KEY:
    resp = client.chat.completions.create(model=MODEL, messages=body["messages"], temperature=0)
    print("type:", type(resp).__name__)
    print("role:", resp.choices[0].message.role)
    print("content:", resp.choices[0].message.content)
    print("total_tokens:", resp.usage.total_tokens)
    sdk_dict = resp.model_dump()            # the typed object, back as a plain dict
else:
    print("skipped (no key): the video printed ChatCompletion, the same sentence, 210 tokens")
    sdk_dict = VIDEO_RESPONSE

type: ChatCompletion
role: assistant
content: It means that nearly one in four checkout attempts are failing, indicating a significant issue that needs immediate investigation.
total_tokens: 210


In [11]:
# Check: the SDK reads the very same JSON fields as our raw request.
print("same top-level fields we use:", {"choices", "usage"} <= set(sdk_dict) and {"choices", "usage"} <= set(data))
print("same content:", sdk_dict["choices"][0]["message"]["content"] == data["choices"][0]["message"]["content"])
print("same total tokens:", sdk_dict["usage"]["total_tokens"] == data["usage"]["total_tokens"])

same top-level fields we use: True
same content: True
same total tokens: True


In the video all three lines were `True`. At temperature 0 the wording *usually* repeats, but not always (lesson 1.2),
so a `False` on the content line is possible and not a bug.

| | Raw `httpx` | `openai` SDK |
|---|---|---|
| what travels | the same JSON | the same JSON |
| you build | URL, headers, body | nothing: one method call |
| you get | a plain dict | a typed object (editor completion) |
| retries | write them yourself (lesson 1.4) | built in (`max_retries`) |

## 6. Roles change behaviour, and where this shows up in agents
### 6.1 Same alert, two system messages

In [12]:
# calls the API (2 requests)
QUESTION = "Alert: checkout error rate is 23%. How bad is it?"
for system in ["Answer in one sentence.", "Answer with exactly one word: CRITICAL, HIGH or LOW."]:
    m = [{"role": "system", "content": system}, {"role": "user", "content": QUESTION}]
    out = ask(m) if HAVE_KEY else "(skipped)"
    print(f"system={system[:30]!r:34} -> {out}")

system='Answer in one sentence.'          -> A 23 % checkout error rate is alarmingly high, indicating a critical problem that likely requires immediate investigation and remediation.


system='Answer with exactly one word: '   -> HIGH


Video: one sentence ("A 23 % checkout error rate is alarmingly high ...") and then the single word `HIGH`.
Same question, different rules, different behaviour: that is the job of the system role.

### 6.2 An agent's memory is this list
Nightwatch (chapter 3) appends to its message list at every step and re-sends the whole list. The cell below replays
one recorded Nightwatch incident from the course cache (**no API call, no key needed**) and prints how long the list
was at each call.

In [13]:
try:
    import agent_v0
    from brightcart.incidents import generate

    sizes, last = [], []
    real_chat = agent_v0.chat
    def spy(body, mode="replay", sample=0):          # same call, but note the list length first
        sizes.append(len(body["messages"]))
        last[:] = [m["role"] for m in body["messages"]]
        return real_chat(body, "replay", sample)
    agent_v0.chat = spy
    inc = next(x for x in generate() if x.id == "bad_deploy-2")
    run = agent_v0.run(inc, mode="replay")
    agent_v0.chat = real_chat
    print("messages sent at each step:", sizes)
    print("roles in the last call:", last)
except (ImportError, LookupError) as e:
    print("replay not available here:", type(e).__name__, e)

messages sent at each step: [2, 4, 6, 8, 10, 12, 13, 15, 17]
roles in the last call: ['system', 'user', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'user', 'assistant', 'tool', 'assistant', 'tool']


The list grows by two at almost every step (the assistant's tool call + the tool's result). Forget one append and
the agent loses its own last step, so it loops or repeats a write.

## 7. Traps
**Trap 1: appending the new question but not the assistant's reply.** The history has a hole: two user messages in a row.

In [14]:
holey = first + second          # forgot the assistant reply in between
roles = [m["role"] for m in holey]
try:
    assert all(not (a == b == "user") for a, b in zip(roles, roles[1:])), f"two user messages in a row: {roles}"
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: two user messages in a row: ['user', 'user']


**Trap 2: rules in only one call.** In the video, call 1 had the one-word rule as a system message and said `HIGH`.
Call 2 sent only a new user question, so the rule was not there: the reply was **273 words** with tables.
Every call needs its own system message.

In [15]:
rule = {"role": "system", "content": "Answer with exactly one word: CRITICAL, HIGH or LOW."}
call_1 = [rule, {"role": "user", "content": QUESTION}]
call_2 = [{"role": "user", "content": "Alert: search latency is 900 ms. How bad is it?"}]
try:
    assert any(m["role"] == "system" for m in call_2), "call 2 has no system message, so the rule is gone"
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: call 2 has no system message, so the rule is gone


**Trap 3: pasting the key into your code.** One shared file leaks it. A tiny check you can run on any code text:

In [16]:
bad_code = 'client = OpenAI(api_key="paste-your-real-key-here")'
good_code = 'client = OpenAI(api_key=os.environ["GROQ_API_KEY"])'
for name, src in [("bad", bad_code), ("good", good_code)]:
    try:
        assert "api_key=\"" not in src, "a literal key string in the code"
        print(name, "-> OK, the key comes from the environment")
    except AssertionError as e:
        print(name, "-> AssertionError:", e)

bad -> AssertionError: a literal key string in the code
good -> OK, the key comes from the environment


**Trap 4: reading the wrong field.** The text sits in `choices`, item 0, `message`, `content`.

In [17]:
for path in ['data["content"]', 'data["choices"]["message"]', 'data["choices"][0]["message"]["content"]']:
    try:
        print(path, "->", eval(path)[:60])
    except (KeyError, TypeError) as e:
        print(path, "->", type(e).__name__, e)

data["content"] -> KeyError 'content'
data["choices"]["message"] -> TypeError list indices must be integers or slices, not str
data["choices"][0]["message"]["content"] -> It means that nearly one in four checkout attempts are faili


## 8. Exercises
None of the check cells call the API: they test your code with a fake model.

In [18]:
def check(n, test):
    """Run the asserts in test(). Print one clear line instead of stopping the notebook."""
    try:
        test()
    except AssertionError as e:
        print(f"✗ Exercise {n} failed: {e}")
    except Exception as e:
        print(f"✗ Exercise {n} failed: {type(e).__name__}: {e}")
    else:
        print(f"✓ Exercise {n} passed")

### Exercise 1 (from the video): system message + a fourth question
Write `run_demo(ask)`. It starts the history with a system message, then for each of the three `QUESTIONS`:
append the question as a `user` message, **print `len(history)`**, call `ask(history)`, and append the reply as an
`assistant` message. Return the history. (The check passes a fake `ask`, so no API is used.)

In [19]:
SYSTEM_EX = "You are Nightwatch, the on-call assistant for the shop Brightcart."
QUESTIONS = ["The failing service is called payments. Remember that.",
             "Which service did I say is failing? Answer with one word, or 'unknown'.",
             "And what did I ask you to remember? One word."]

def run_demo(ask):
    history = [{"role": "system", "content": SYSTEM_EX}]
    # YOUR CODE HERE
    return history

In [20]:
def test_1():
    sent = []
    def fake_ask(messages):
        sent.append([m["role"] for m in messages])
        return f"reply {len(sent)}"
    h = run_demo(fake_ask)
    assert [len(s) for s in sent] == [2, 4, 6], f"calls should send 2, 4, 6 messages, got {[len(s) for s in sent]}"
    assert all(s[0] == "system" for s in sent), "every call must start with the system message"
    assert [m["role"] for m in h] == ["system", "user", "assistant", "user", "assistant", "user", "assistant"], \
        f"roles are {[m['role'] for m in h]}"
    assert h[-1]["content"] == "reply 3", "append each reply as an assistant message"

check(1, test_1)

✗ Exercise 1 failed: calls should send 2, 4, 6 messages, got []


In [21]:
# calls the API (3 requests): optional, run your demo for real once the check passes
demo = run_demo(ask) if HAVE_KEY else None
if demo is not None and len(demo) == 1:
    print("write run_demo in Exercise 1 first")
elif demo:
    for m in demo[1:]:
        print(f"{m['role']:9} {m['content'][:80]}")
else:
    print("skipped (no key)")

write run_demo in Exercise 1 first


### Exercise 2: read the right field
Write `get_text(data)` that returns the reply's text from a raw response dict.

In [22]:
def get_text(data):
    # YOUR CODE HERE
    return None

In [23]:
def test_2():
    assert get_text(VIDEO_RESPONSE).startswith("It means that nearly one in four"), f"got {get_text(VIDEO_RESPONSE)!r}"
    other = {"choices": [{"message": {"role": "assistant", "content": "HIGH"}}], "usage": {}}
    assert get_text(other) == "HIGH", f"got {get_text(other)!r}"

check(2, test_2)

✗ Exercise 2 failed: AttributeError: 'NoneType' object has no attribute 'startswith'


### Exercise 3: build a request body
Write `make_body(system, user, temperature=0)` that returns a body like the one in 4.1: keys `model`, `messages`
(system first, then user) and `temperature`.

In [24]:
def make_body(system, user, temperature=0):
    # YOUR CODE HERE
    return {}

In [25]:
def test_3():
    b = make_body("Answer in one sentence.", "Alert: disk is 95% full.")
    assert set(b) == {"model", "messages", "temperature"}, f"keys are {set(b)}"
    assert b["model"] == MODEL, "use the course model"
    assert [m["role"] for m in b["messages"]] == ["system", "user"], "system first, then user"
    assert b["messages"][1]["content"] == "Alert: disk is 95% full.", "the user text goes in the user message"
    assert make_body("s", "u", temperature=1)["temperature"] == 1, "pass the temperature through"
    assert make_body(SYSTEM, ALERT) == body, "with SYSTEM and ALERT it should equal body from 4.1"

check(3, test_3)

✗ Exercise 3 failed: keys are set()


### Exercise 4: find the hole
Write `find_hole(messages)` that returns the index of the first message that is a `user` message right after another
`user` message (a missing assistant reply), or `None` if there is no hole.

In [26]:
def find_hole(messages):
    # YOUR CODE HERE
    return "not written yet"

In [27]:
def test_4():
    assert find_hole(history) is None, "the hook's 3-message history has no hole"
    assert find_hole(holey) == 1, f"first + second has a hole at index 1, got {find_hole(holey)}"
    sys_first = [{"role": "system", "content": "x"}] + first + [{"role": "assistant", "content": "ok"}] + second + second
    assert find_hole(sys_first) == 4, f"expected 4, got {find_hole(sys_first)}"
    assert find_hole([]) is None, "an empty list has no hole"

check(4, test_4)

✗ Exercise 4 failed: the hook's 3-message history has no hole


## Solutions (try first!)
Run a solution cell, then run its check cell above again to see it pass.

### Solution 1

In [28]:
SYSTEM_EX = "You are Nightwatch, the on-call assistant for the shop Brightcart."
QUESTIONS = ["The failing service is called payments. Remember that.",
             "Which service did I say is failing? Answer with one word, or 'unknown'.",
             "And what did I ask you to remember? One word."]

def run_demo(ask):
    history = [{"role": "system", "content": SYSTEM_EX}]
    for q in QUESTIONS:
        history.append({"role": "user", "content": q})
        print("sending", len(history), "messages")
        reply = ask(history)
        history.append({"role": "assistant", "content": reply})
    return history

### Solution 2

In [29]:
def get_text(data):
    return data["choices"][0]["message"]["content"]

### Solution 3

In [30]:
def make_body(system, user, temperature=0):
    return {"model": MODEL,
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
            "temperature": temperature}

### Solution 4

In [31]:
def find_hole(messages):
    for i in range(1, len(messages)):
        if messages[i]["role"] == "user" and messages[i - 1]["role"] == "user":
            return i
    return None